In [2]:
def b_tree_search(x, k):
    i = 0

    while i < x.n and k > x.keys[i]:
        i += 1

    if i < x.n and k == x.keys[i]:
        return (x, i)

    if x.leaf:
        return None

    # disk_read(x.children[i])  # Stub: would load child from disk in real implementation
    return b_tree_search(x.children[i], k)

In [33]:
class Node:
    def __init__(self, t, leaf=False):
        self.t = t
        self.leaf = leaf
        self.keys = []
        self.children = []
        self.n = len(self.keys)

    # @property
    # def n(self):
    #     return len(self.keys)        # Number of keys

In [34]:
def allocate_node(t):
    return Node(t)

In [35]:
def b_tree_create(t):
    x = allocate_node(t)
    x.leaf = True
    x.n = 0
    x.keys = []
    x.children = []
    # disk_write(x)  # Stub for disk-based implementation
    t.root = x

In [36]:
class BTree:
    def __init__(self, t):
        self.t = t
        self.root = None

In [37]:
t = BTree(2)   # minimum degree = 2
b_tree_create(t)

In [39]:
"""
1. y = x.children[i]           # Identify full child to split
2. z = allocate_node(t, leaf=y.leaf)   # Create new node for right half
3. z.keys = y.keys[t:]          # Move right half of keys to z
4. z.children = y.children[t:] if not y.leaf else []  # Move right half children if internal
5. middle_key = y.keys[t-1]     # Middle key to push up to parent
6. y.keys = y.keys[:t-1]        # Keep left half in y
7. y.children = y.children[:t] if not y.leaf else []  # Shrink y's children
8. x.children.insert(i+1, z)   # Insert z as new child in parent
9. x.keys.insert(i, middle_key) # Insert middle key into parent
10. x.n += 1                     # Update parent key count

BEFORE split:
y: [10, 15, 18]  ← FULL
x: []

AFTER split:
      15
     /  \
[10]       [18]

split into:
[t-1 keys]  +  [1 middle]  +  [t-1 keys]

Think of x as a container with multiple pockets (children).
i tells you which pocket is full.
t tells you how big each pocket is allowed to be.

Non-Leaf Node (t=2)

Before split:
x.keys = [20]
x.children = [y]
y.keys = [10, 15, 18]  # FULL
y.children = [c0, c1, c2, c3]
y.leaf = False

After split:
       [15, 20]
      /       \
y:[10]        z:[18]
/  \          /  \
c0  c1       c2  c3
"""
def b_tree_split_child(x, i, t):
    y = x.children[i]
    z = allocate_node(t, leaf=y.leaf)

    # Copy last (t-1) keys of y into z
    z.keys = y.keys[t:]
    z.children = y.children[t:] if not y.leaf else []
    
    z.n = t - 1

    middle_key = y.keys[t - 1]
    y.keys = y.keys[:t - 1]
    y.children = y.children[:t] if not y.leaf else []
    y.n = t - 1

    x.children.insert(i + 1, z)

    x.keys.insert(i, middle_key)
    x.n += 1

    # disk_write(y)
    # disk_write(z)
    # disk_write(x)

In [ ]:
"""
t = 2
tree = BTree(t)
b_tree_create(tree)
for key in [10, 20, 5, 6, 12, 30, 7, 17]:
    b_tree_insert(tree, key)
"""

In [40]:
def b_tree_split_root(t):
    s = allocate_node(t)
    s.leaf = False
    s.n = 0
    s.children = [t.root]  # wrap root in a list
    t.root = s
    b_tree_split_child(s, 0, t)
    return s

In [44]:
def b_tree_insert_nonfull(x, k, t):
    i = x.n - 1  # start from last key
    if x.leaf:
        x.keys.append(0)  # temporary space
        while i >= 0 and k < x.keys[i]:
            x.keys[i+1] = x.keys[i]  # shift right
            i -= 1
        x.keys[i+1] = k
        x.n += 1
    else:
        while i >= 0 and k < x.keys[i]:
            i -= 1
        i += 1
        if x.children[i].n == 2*t - 1:
            b_tree_split_child(x, i, t)
            if k > x.keys[i]:
                i += 1
        b_tree_insert_nonfull(x.children[i], k, t)

In [46]:
def b_tree_insert(t, k):
    r = t.root
    if r.n == 2*t - 1:
        s = b_tree_split_root(t)
        b_tree_insert_nonfull(s, k)
    else:
        b_tree_insert_nonfull(r, k)

In [47]:
"""
B-Tree Deletion, many cases:
Only worry when a node is "too small"
If ANY sibling has ≥ t keys → BORROW
Else → MERGE

Borrowing is possible only if an adjacent sibling has at least t keys.
If all siblings have exactly t−1 keys, borrowing is impossible and a merge must be performed.

During a merge, the key from the parent that separates the two sibling nodes is moved down into the merged node,
combining both children and reducing the parent’s number of keys.

1. Not found → stop

2. Key in leaf → delete directly

3. Key in internal:
   - If left child ≥ t → use predecessor
   - Else if right child ≥ t → use successor
   - Else → merge

4. While descending:
   - If child has t-1 keys:
       → borrow from sibling if possible
       → else merge

Sibling has extra keys → we BORROW
No sibling can lend → must merge

AN EASY PATTERN TO REMEMBER:
BORROW if possible
MERGE if forced
REPLACE if internal

LEAF → delete
INTERNAL → replace
UNDERFLOW → borrow
NO BORROW → merge

A merge occurs during B-tree deletion when the target node and its siblings all have the minimum number of keys,
making borrowing impossible in this case, the node is merged with a sibling and a key from the parent.
"""

'\nB-Tree Deletion, many cases:\nOnly worry when a node is "too small"\nIf ANY sibling has ≥ t keys → BORROW\nElse → MERGE\n\nBorrowing is possible only if an adjacent sibling has at least t keys.\nIf all siblings have exactly t−1 keys, borrowing is impossible and a merge must be performed.\n\nDuring a merge, the key from the parent that separates the two sibling nodes is moved down into the merged node,\ncombining both children and reducing the parent’s number of keys.\n\n1. Not found → stop\n\n2. Key in leaf → delete directly\n\n3. Key in internal:\n   - If left child ≥ t → use predecessor\n   - Else if right child ≥ t → use successor\n   - Else → merge\n\n4. While descending:\n   - If child has t-1 keys:\n       → borrow from sibling if possible\n       → else merge\n\nSibling has extra keys → we BORROW\nNo sibling can lend → must merge\n\nAN EASY PATTERN TO REMEMBER:\nBORROW if possible\nMERGE if forced\nREPLACE if internal\n\nLEAF → delete\nINTERNAL → replace\nUNDERFLOW → borrow\n

In [48]:
"""
Keep the top two blocks of the stack in memory. Perform operations on the top block. When the top block becomes full or empty,
shift blocks between memory and disk while maintaining the invariant that the top two blocks are in memory.
This ensures that disk accesses occur only once every m operations, giving an amortized cost of 
O(1/m) disk accesses per operation and O(1) CPU time.
"""

'\nKeep the top two blocks of the stack in memory. Perform operations on the top block. When the top block becomes full or empty,\nshift blocks between memory and disk while maintaining the invariant that the top two blocks are in memory.\nThis ensures that disk accesses occur only once every m operations, giving an amortized cost of \nO(1/m) disk accesses per operation and O(1) CPU time.\n'